In [17]:
print("ram ram")

ram ram


In [18]:
# # Rule-based agentic pipeline generation for MES APIs

# This notebook shows a deterministic architecture for onboarding MES backend APIs into agentic pipelines.

# Principles:
# - Discover Swagger/OpenAPI APIs
# - Keep only APIs where metadata says `agent_accessible = true` and `auto_register = true`
# - Group by `business_domain`
# - Create one agent per domain
# - Build pipelines from ordered tool metadata
# - Use an LLM only for final reasoning over retrieved results, not for building the pipeline

In [19]:
import json
import requests
import pandas as pd
import urllib3
from requests.exceptions import ConnectionError, Timeout, RequestException

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://localhost:7204"
SWAGGER_URL = f"{BASE_URL}/swagger/v1/swagger.json"
TOKEN = None  # set this if your backend requires auth

headers = {"Accept": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"


def fetch_swagger(url, headers=None, verify=False):
    response = requests.get(url, headers=headers, verify=verify, timeout=15)
    response.raise_for_status()
    return response.json()

try:
    swagger = fetch_swagger(SWAGGER_URL, headers=headers)
    print("Swagger loaded successfully")
except Exception as e:
    print(f"Could not load swagger: {e}")
    swagger = {"paths": {}}

Swagger loaded successfully


In [20]:
def parse_json_if_possible(value):
    if not value:
        return None
    if isinstance(value, dict) or isinstance(value, list):
        return value
    try:
        return json.loads(value)
    except Exception:
        return value

rows = []
for path, methods in swagger.get("paths", {}).items():
    for method, details in methods.items():
        if method.lower() not in {"get", "post", "put", "patch", "delete"}:
            continue

        metadata = {}
        description = parse_json_if_possible(details.get("description"))
        if isinstance(description, dict):
            metadata.update(description)
        elif isinstance(description, str):
            metadata["description_text"] = description

        ext = details.get("x-metadata", {})
        if isinstance(ext, dict):
            metadata.update(ext)

        tool_name = (
            metadata.get("tool_name")
            or details.get("operationId")
            or f"{method.upper()}_{path.replace('/', '_')}"
        )

        rows.append({
            "tool_name": tool_name,
            "path": path,
            "method": method.upper(),
            "description": metadata.get("description") or details.get("summary") or "",
            "business_domain": metadata.get("business_domain") or "Unassigned",
            "recommended_agents": metadata.get("recommended_agents", []),
            "restricted_agents": metadata.get("restricted_agents", []),
            "agent_accessible": bool(metadata.get("agent_accessible", True)),
            "auto_register": bool(metadata.get("auto_register", True)),
            "operation_type": metadata.get("operation_type") or method.upper(),
            "risk_level": metadata.get("risk_level") or "MEDIUM",
            "priority": int(metadata.get("priority", 100)),
        })

backend_api_df = pd.DataFrame(rows)
backend_api_df.head()

,tool_name,path,method,description,business_domain,recommended_agents,restricted_agents,agent_accessible,auto_register,operation_type,risk_level,priority
0,get_status,/api/v1/account,GET,Executes GetStatus. Usable by AccountApi agents.,Unassigned,[],[],True,True,GET,MEDIUM,100
1,create_user_profile,/api/v1/account/user-creation,POST,Executes CreateUserProfile. Usable by AccountA...,Unassigned,[],[],False,True,POST,MEDIUM,100
2,get_roles,/api/v1/account/roles,GET,Executes GetRoles. Usable by AccountApi agents.,Unassigned,[],[],True,True,GET,MEDIUM,100
3,get_menus,/api/v1/account/menu,GET,Executes GetMenus. Usable by AccountApi agents.,Unassigned,[],[],True,True,GET,MEDIUM,100
4,user_by_id,/api/v1/account/user-by-id/{id},GET,Executes UserById. Usable by AccountApi agents.,Unassigned,[],[],True,True,GET,MEDIUM,100


In [21]:
usable_api_df = backend_api_df[
    (backend_api_df["agent_accessible"] == True) &
    (backend_api_df["auto_register"] == True)
].copy()

usable_api_df = usable_api_df.sort_values(["business_domain", "priority", "tool_name"]).reset_index(drop=True)
print(f"Total discovered APIs: {len(backend_api_df)}")
print(f"Usable APIs for agentic pipelines: {len(usable_api_df)}")

usable_api_df[["tool_name", "business_domain", "operation_type", "priority", "agent_accessible", "auto_register"]].head(20)

Total discovered APIs: 670
Usable APIs for agentic pipelines: 362


,tool_name,business_domain,operation_type,priority,agent_accessible,auto_register
0,GET__api_v1_wiptraceability_getIncompleteJumbo...,Unassigned,GET,100,True,True
1,bom,Unassigned,GET,100,True,True
2,check_bag_status,Unassigned,GET,100,True,True
3,check_r_m_bag_staus,Unassigned,GET,100,True,True
4,download,Unassigned,GET,100,True,True
5,exceptions,Unassigned,GET,100,True,True
6,fore_cast_date_validation,Unassigned,GET,100,True,True
7,gate_by_locatio,Unassigned,GET,100,True,True
8,generate_q_r_code,Unassigned,GET,100,True,True
9,get,Unassigned,GET,100,True,True


In [22]:
def build_agent_definitions(api_df):
    agents = []
    for domain, group in api_df.groupby("business_domain", sort=True):
        if group.empty:
            continue
        agents.append({
            "agent_name": f"{domain} Agent",
            "business_domain": domain,
            "tool_count": len(group),
            "tools": group["tool_name"].tolist(),
            "registration_mode": "rule_based",
        })
    return pd.DataFrame(agents)

agent_registry_df = build_agent_definitions(usable_api_df)
agent_registry_df

,agent_name,business_domain,tool_count,tools,registration_mode
0,Unassigned Agent,Unassigned,362,[GET__api_v1_wiptraceability_getIncompleteJumb...,rule_based


In [23]:
def build_rule_based_pipeline_registry(api_df):
    pipeline_rows = []
    pipeline_graph = {}

    for domain, group in api_df.groupby("business_domain", sort=True):
        if group.empty:
            continue

        ordered_group = group.sort_values(["priority", "tool_name"], ascending=[True, True]).reset_index(drop=True)
        selected_tools = ordered_group.head(5).copy()

        agent_name = f"{domain} Agent"
        pipeline_name = f"{domain.lower().replace(' ', '_')}_pipeline"

        graph_steps = []
        for idx, row in selected_tools.iterrows():
            step_name = f"step_{idx + 1}"
            step_id = f"{pipeline_name}_{step_name}"
            depends_on = [] if idx == 0 else [f"{pipeline_name}_step_{idx}"]
            graph_steps.append({
                "step_id": step_id,
                "step_name": step_name,
                "tool_name": row["tool_name"],
                "operation_type": row["operation_type"],
                "priority": int(row["priority"]),
                "depends_on": depends_on,
            })

            pipeline_rows.append({
                "pipeline_name": pipeline_name,
                "agent_name": agent_name,
                "business_domain": domain,
                "step_order": idx + 1,
                "step_name": step_name,
                "tool_name": row["tool_name"],
                "path": row["path"],
                "operation_type": row["operation_type"],
                "priority": int(row["priority"]),
                "uses_llm": False,
                "reason": "metadata_ordered_rule_based",
            })

        pipeline_graph[agent_name] = {
            "business_domain": domain,
            "pipeline_name": pipeline_name,
            "steps": graph_steps,
        }

    return pd.DataFrame(pipeline_rows), pipeline_graph


pipeline_registry_df, pipeline_graph = build_rule_based_pipeline_registry(usable_api_df)
pipeline_registry_df.head(15)

,pipeline_name,agent_name,business_domain,step_order,step_name,tool_name,path,operation_type,priority,uses_llm,reason
0,unassigned_pipeline,Unassigned Agent,Unassigned,1,step_1,GET__api_v1_wiptraceability_getIncompleteJumbo...,/api/v1/wiptraceability/getIncompleteJumboBag/...,GET,100,False,metadata_ordered_rule_based
1,unassigned_pipeline,Unassigned Agent,Unassigned,2,step_2,bom,/api/v1/planning/bom/{productCode},GET,100,False,metadata_ordered_rule_based
2,unassigned_pipeline,Unassigned Agent,Unassigned,3,step_3,check_bag_status,/api/v1/wiptraceability/check-bag-status/{bagN...,GET,100,False,metadata_ordered_rule_based
3,unassigned_pipeline,Unassigned Agent,Unassigned,4,step_4,check_r_m_bag_staus,/api/v1/wiptraceability/rm-bag/{bagName},GET,100,False,metadata_ordered_rule_based
4,unassigned_pipeline,Unassigned Agent,Unassigned,5,step_5,download,/api/v1/attachment/downaload-by-id/{id},GET,100,False,metadata_ordered_rule_based


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

In [27]:
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "manufacturing_ai",
    "user": "postgres",
    "password": "0987654321"
}

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL)

In [28]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print(result.fetchone()[0])
        print("\n✅ PostgreSQL Connected Successfully")
except Exception as e:
    print(e)

PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit

✅ PostgreSQL Connected Successfully


In [29]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)
tables

,table_name
0,agent_execution_logs
1,agent_tool_mapping
2,agentic_pipline
3,alert_master
4,api_logs
5,app_authentication
6,app_connection_table
7,app_table
8,app_version_history
9,approval_history


In [30]:
query = """
SELECT
table_name,
column_name,
data_type
FROM information_schema.columns
WHERE table_schema='public'
ORDER BY table_name, ordinal_position;
"""

columns = pd.read_sql(query, engine)
columns

,table_name,column_name,data_type
0,agent_execution_logs,execution_id,uuid
1,agent_execution_logs,agentic_id,character varying
2,agent_execution_logs,started_at,timestamp with time zone
3,agent_execution_logs,completed_at,timestamp with time zone
4,agent_execution_logs,status,character varying
...,...,...,...
173,workflow_history,input_state,jsonb
174,workflow_history,output_state,jsonb
175,workflow_history,approval_status,character varying
176,workflow_history,customer_id,integer


In [31]:
query = """
SELECT COUNT(*)
FROM information_schema.tables
WHERE table_schema='public';
"""

pd.read_sql(query, engine)

,count
0,26
